# 01 Load and profile

**What this notebook does.** It reads the six Favorita files, converts the large training table to Parquet in chunks and profiles every table. Nothing is cleaned or changed here; cleaning starts in notebook 02.

## Data dictionary

| Table | One row is | Rows | Used for |
|---|---|---|---|
| `train` | one store, item and date with a recorded sale | 125,497,040 | target `unit_sales`, `onpromotion` |
| `stores` | one store | 54 | city, state, type, cluster |
| `items` | one item | 4,100 | family, class, perishable flag |
| `oil` | one day | 1,218 | daily oil price (43 missing) |
| `holidays_events` | one holiday or event | 350 | calendar information |
| `transactions` | one store and date | 83,488 | number of receipts per day |

**Target.** `unit_sales` is the number of units sold (items or kilograms) for one store, item and day. A row exists only when something was sold or returned, so the data contain no rows with zero sales. The period is 1 January 2013 to 15 August 2017.

**Important.** `transactions` is only known after a day has ended, so it is used for analysis only and never as a feature for a forecast issued before that day.

## 1. Profile of the five small tables

**Purpose.** Check the structure, data types, missing values and duplicates of the five dimension tables before they are used.

**Method.** Each CSV file is read in full. For each table the shape, data types, number of missing values, number of duplicate rows and the first three rows are shown.

**Output.** One short report per table.

**Findings.** Only `oil` has missing values (43 of 1,218 prices, including the first day, 2013-01-01). No table has duplicate rows. Dates are stored as text in `oil`, `holidays_events` and `transactions` and are converted in notebook 02. `oil` has 1,218 rows, fewer than the days with sales, because it has no prices for weekends; this is handled in notebook 02.

In [5]:
import pandas as pd
from pathlib import Path

BASE = Path.home() / "Projects/bigdata"
RAW = BASE / "raw"

tables = {}
for name in ["stores", "items", "oil", "holidays_events", "transactions"]:
    df = pd.read_csv(RAW / f"{name}.csv")
    tables[name] = df
    print(f"\n=== {name}: {df.shape[0]:,} rows x {df.shape[1]} cols ===")
    print(df.dtypes.to_string())
    nulls = df.isna().sum()
    print("nulls:", nulls[nulls > 0].to_dict() if (nulls > 0).any() else "none")
    print("duplicate rows:", int(df.duplicated().sum()))
    display(df.head(3))


=== stores: 54 rows x 5 cols ===
store_nbr     int64
city         object
state        object
type         object
cluster       int64
nulls: none
duplicate rows: 0


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8



=== items: 4,100 rows x 4 cols ===
item_nbr       int64
family        object
class          int64
perishable     int64
nulls: none
duplicate rows: 0


,item_nbr,family,class,perishable
0,96995,GROCERY I,1093,0
1,99197,GROCERY I,1067,0
2,103501,CLEANING,3008,0



=== oil: 1,218 rows x 2 cols ===
date           object
dcoilwtico    float64
nulls: {'dcoilwtico': 43}
duplicate rows: 0


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97



=== holidays_events: 350 rows x 6 cols ===
date           object
type           object
locale         object
locale_name    object
description    object
transferred      bool
nulls: none
duplicate rows: 0


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False



=== transactions: 83,488 rows x 3 cols ===
date            object
store_nbr        int64
transactions     int64
nulls: none
duplicate rows: 0


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358


## 2. Read the training table in chunks

**Purpose.** `train.csv` (4.7 GB, 125 million rows) is too large to handle comfortably in one step, so it is converted to compact Parquet files that the next notebooks can read quickly.

**Method.** The file is read in chunks of 10 million rows with compact data types (`int8` for the store, `int32` for the item and the id, `float32` for sales, nullable boolean for the promotion flag). While each chunk is written to `interim/train/`, the number of rows, negative sales, missing promotion values and the date range are counted. Old Parquet parts are removed first so that a re-run cannot leave stale files.

**Output.** 13 Parquet files in `interim/train/` and a summary dictionary.

**Findings.** 125,497,040 rows (equal to the line count of the file minus the header), 7,795 negative sales (0.006%), 21,657,651 missing `onpromotion` values (17.3%) and a period from 2013-01-01 to 2017-08-15.

**Limitation.** The counts are made chunk by chunk, so duplicates across chunks are not checked here; the check of the key (date, store, item) is done in notebook 02.

In [6]:
import pandas as pd
from pathlib import Path

BASE = Path.home() / "Projects/bigdata"
RAW = BASE / "raw"
OUT = BASE / "interim" / "train"
OUT.mkdir(parents=True, exist_ok=True)
for old in OUT.glob("part-*.parquet"):
    old.unlink()

dtypes = {
    "id": "int32",
    "store_nbr": "int8",
    "item_nbr": "int32",
    "unit_sales": "float32",
    "onpromotion": "boolean",
}

stats = {"rows": 0, "neg_sales": 0, "null_promo": 0,
         "date_min": None, "date_max": None}

reader = pd.read_csv(RAW / "train.csv", dtype=dtypes,
                     parse_dates=["date"], chunksize=10_000_000)

for i, chunk in enumerate(reader):
    stats["rows"] += len(chunk)
    stats["neg_sales"] += int((chunk["unit_sales"] < 0).sum())
    stats["null_promo"] += int(chunk["onpromotion"].isna().sum())
    dmin, dmax = chunk["date"].min(), chunk["date"].max()
    stats["date_min"] = dmin if stats["date_min"] is None else min(stats["date_min"], dmin)
    stats["date_max"] = dmax if stats["date_max"] is None else max(stats["date_max"], dmax)
    chunk.to_parquet(OUT / f"part-{i:03d}.parquet", index=False)
    print(f"chunk {i} done")

print(stats)

chunk 0 done
chunk 1 done
chunk 2 done
chunk 3 done
chunk 4 done
chunk 5 done
chunk 6 done
chunk 7 done
chunk 8 done
chunk 9 done
chunk 10 done
chunk 11 done
chunk 12 done
{'rows': 125497040, 'neg_sales': 7795, 'null_promo': 21657651, 'date_min': Timestamp('2013-01-01 00:00:00'), 'date_max': Timestamp('2017-08-15 00:00:00')}


## 3. Coverage of stores, items and dates

**Purpose.** Check that the training table and the dimension tables describe the same stores and items, so that joins in notebook 02 do not lose or invent rows.

**Method.** Read the store, item and date columns of the Parquet files and compare the sets of stores and items with `stores.csv` and `items.csv`.

**Output.** Counts of stores, items and dates, and the differences between train and the dimension tables.

**Limitation.** Having an item in `items.csv` does not mean it was ever sold: items without any record in train cannot be forecast and are dropped in notebook 02.

In [7]:
   import pandas as pd
   from pathlib import Path

   BASE = Path.home() / "Projects/bigdata"
   RAW = BASE / "raw"
   OUT = BASE / "interim" / "train"

   keys = pd.read_parquet(OUT, columns=["date", "store_nbr", "item_nbr"])
   stores = pd.read_csv(RAW / "stores.csv")
   items = pd.read_csv(RAW / "items.csv")

   s_train, i_train = set(keys["store_nbr"].unique()), set(keys["item_nbr"].unique())
   s_meta, i_meta = set(stores["store_nbr"]), set(items["item_nbr"])

   print(f"rows: {len(keys):,} | distinct dates: {keys['date'].nunique():,} "
         f"({keys['date'].min().date()} to {keys['date'].max().date()})")
   print(f"stores: {len(s_train)} in train, {len(s_meta)} in stores.csv | "
         f"in train but not in stores.csv: {len(s_train - s_meta)} | in stores.csv but not in train: {len(s_meta - s_train)}")
   print(f"items:  {len(i_train):,} in train, {len(i_meta):,} in items.csv | "
         f"in train but not in items.csv: {len(i_train - i_meta)} | in items.csv but not in train: {len(i_meta - i_train)}")

rows: 125,497,040 | distinct dates: 1,684 (2013-01-01 to 2017-08-15)
stores: 54 in train, 54 in stores.csv | in train but not in stores.csv: 0 | in stores.csv but not in train: 0
items:  4,036 in train, 4,100 in items.csv | in train but not in items.csv: 0 | in items.csv but not in train: 64


**Findings.** The training table has 125,497,040 rows on 1,684 distinct dates (1 January 2013 to 15 August 2017). All 54 stores of `stores.csv` appear in train. Train contains 4,036 items, all of which are in `items.csv`; the 64 items of `items.csv` that are never sold in train cannot be forecast and are dropped in notebook 02. 1,684 days is fewer than the 1,688 calendar days of the period because 25 December is missing in 2013 to 2016 (stores closed).